## 1. Introduction



#2. Data Loading and Preprocessing

## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# Text processing
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, FreqDist, ngrams
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# Dimensionality reduction
from sklearn.decomposition import PCA
# import umap  # Uncomment if needed

# Clustering
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

# Visualization
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from collections import Counter

sns.set(style="whitegrid", context="notebook")


## Loading dataset

In [ ]:
df = pd.read_json("/content/yelp_academic_dataset_business.json", lines=True)
df.head()

## Dataset Overview & Missing Value Analysis

In [ ]:
print(df.shape)
print(df.info())
print("\nMissing values:\n", df.isna().sum())

* The dataset contains 150,346 rows and 14 columns.
* All essential fields (business_id, name, address, city, state, postal_code, latitude, longitude, stars, review_count, is_open) have 0 missing values.
* Only optional metadata fields contain missing data:
**attributes**: 13,744 missing
**categories**: 103 missing
**hours**: 23,223 missing
* Overall, the dataset is clean and complete for core analysis, with missingness limited to secondary fields.

In [ ]:
pip install cartopy

#3. Exploratory Data Analysis (EDA)

In [ ]:
# Star ratings distribution
plt.figure(figsize=(8,4))
sns.histplot(df['stars'], bins=10, kde=True)
plt.title("Distribution of Star Ratings")
plt.xlabel('Star Ratings')
plt.ylabel('Frequency')
plt.show()

# Top categories
business_cats = ''.join(df['categories'].astype('str'))
cats = pd.DataFrame(business_cats.split(','), columns=['categories'])
x = cats.categories.value_counts()
x = x.sort_values(ascending=False)
x = x.iloc[:20]

plt.figure(figsize=(16,4))
ax = sns.barplot(x=x.index, y=x.values, alpha=0.8)
plt.title("What are the top categories?", fontsize=25)
locs, labels = plt.xticks()
plt.setp(labels, rotation=80)
plt.ylabel('# businesses', fontsize=12)
plt.xlabel('Category', fontsize=12)
plt.show()

# Top cities
x = df['city'].value_counts()
x = x.sort_values(ascending=False)
x = x.iloc[0:20]

plt.figure(figsize=(16,4))
ax = sns.barplot(x=x.index, y=x.values, alpha=0.8)
plt.title("Which city has the most reviews?")
locs, labels = plt.xticks()
plt.setp(labels, rotation=45)
plt.ylabel('# businesses', fontsize=12)
plt.xlabel('City', fontsize=12)
plt.show()

# Geographic distribution (requires cartopy)
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    plt.figure(figsize=(15, 6))
    ax = plt.axes(projection=ccrs.Orthographic(central_longitude=-50, central_latitude=20))
    ax.set_global()
    ax.set_facecolor("#4a80f5")
    ax.add_feature(cfeature.LAND, facecolor="#bbdaa4")
    ax.add_feature(cfeature.OCEAN, facecolor="#4a80f5")
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="black")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.3, edgecolor="black")

    lons = df["longitude"].tolist()
    lats = df["latitude"].tolist()
    ax.scatter(lons, lats, s=3, color="orange", transform=ccrs.PlateCarree(), zorder=5)
    plt.title("World-wide Yelp Reviews", fontsize=16)
    plt.show()
except ImportError:
    print("Cartopy not installed, skipping geographic visualization")

# Category counts
cats = []
for row in df['categories'].dropna():
    cats.extend([c.strip() for c in row.split(",")])

print("\nTop 20 Categories:")
print(pd.Series(Counter(cats)).sort_values(ascending=False).head(20))

# Review count distribution
sns.histplot(df['review_count'], log_scale=True)
plt.title("Review Count (log scale)")
plt.show()

# Numerical distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

df['latitude'].hist(ax=axes[0,0])
axes[0,0].set_title('Latitude')

df['longitude'].hist(ax=axes[0,1])
axes[0,1].set_title('Longitude')

df['stars'].hist(ax=axes[0,2])
axes[0,2].set_title('Stars')

df['review_count'].hist(ax=axes[1,0])
axes[1,0].set_title('Review Count')

df['is_open'].hist(ax=axes[1,1])
axes[1,1].set_title('Is Open')

# Turn off unused axis
axes[1,2].axis('off')

plt.tight_layout()
plt.show()


* Star ratings are skewed toward higher values, with most businesses rated between 3.5 and 4.5 stars.
* Categories show a heavy concentration in service industries, led by Restaurants, Food, and Shopping.
* Philadelphia, Tucson, Tampa, Indianapolis, and Nashville have the highest number of businesses, indicating major Yelp activity hubs.
* Geographic plots confirm dense clustering around major U.S. metro regions with very few globally distributed points.
* Review counts follow a long-tail distribution, where most businesses have few reviews and only a small fraction receive very high engagement.
* Numerical distributions show strong concentration around key latitude/longitude ranges, and most businesses are marked as open.

#Feature Engineering

In [ ]:
# Parse attributes
def parse_attributes(attr):
    if attr is None:
        return {}
    out = {}
    for k, v in attr.items():
        try:
            parsed = ast.literal_eval(v) if isinstance(v, str) and (v.startswith("{") or v.startswith("u") or v.startswith("'") or v.startswith('"')) else v
        except:
            parsed = v
        out[k] = parsed
    return out

df['parsed_attributes'] = df['attributes'].apply(parse_attributes)

# Flatten attributes
from pandas import json_normalize
attr_df = json_normalize(df['parsed_attributes'])
df = pd.concat([df, attr_df], axis=1)
df.drop(columns=['parsed_attributes', 'attributes'], inplace=True)

# Create feature matrix
num_features = df[['stars', 'review_count', 'latitude', 'longitude']].copy()

# Multi-hot encode categories
df['categories_list'] = df['categories'].fillna("").apply(lambda x: [c.strip() for c in x.split(",")])
mlb = MultiLabelBinarizer()
cat_features = pd.DataFrame(mlb.fit_transform(df['categories_list']),
                            columns=mlb.classes_,
                            index=df.index)

X = pd.concat([num_features, cat_features], axis=1).fillna(0)

# Scale numerical features
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[['stars', 'review_count', 'latitude', 'longitude']] = scaler.fit_transform(
    X[['stars', 'review_count', 'latitude', 'longitude']]
)

#4. K-Means Clustering

In [ ]:
X_geo = df[['latitude', 'longitude']].dropna()
df_geo = df.loc[X_geo.index].copy()
X_geo_scaled = scaler.fit_transform(X_geo)

print(f"Total records for clustering: {len(X_geo_scaled):,}")

SAMPLE_SIZE = min(50000, len(X_geo_scaled))  # Use max 50k for speed
print(f"Using sample of {SAMPLE_SIZE:,} records for k selection")

np.random.seed(42)
sample_idx = np.random.choice(len(X_geo_scaled), SAMPLE_SIZE, replace=False)
X_sample = X_geo_scaled[sample_idx]

# Elbow Method (Manual) - on sample
inertia = []
Ks = range(2, 12)

for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_sample)
    inertia.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(Ks, inertia, marker='o')
plt.title("Elbow Method")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.show()

# Elbow Method (Yellowbrick) - on sample
model = KMeans(random_state=42, n_init=10)
visualizer = KElbowVisualizer(model, k=(2,12))
visualizer.fit(X_sample)
visualizer.show()

best_k_elbow = visualizer.elbow_value_
print(f"Best k from Yellowbrick elbow: {best_k_elbow}")

# Validation metrics - on sample
sil_scores = []
db_scores = []
ch_scores = []

print("\nCalculating validation metrics...")
for k in Ks:
    print(f"  k={k}...", end=" ")
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sample)

    sil_scores.append(silhouette_score(X_sample, labels))
    db_scores.append(davies_bouldin_score(X_sample, labels))
    ch_scores.append(calinski_harabasz_score(X_sample, labels))
    print("✓")

print("Done!")

metrics_df = pd.DataFrame({
    'k': Ks,
    'silhouette': sil_scores,
    'davies_bouldin': db_scores,
    'calinski_harabasz': ch_scores
})

print("\nCluster Validation Metrics:")
print(metrics_df)

# Silhouette scores plot
plt.figure(figsize=(10,5))
plt.plot(Ks, sil_scores, marker='o')
plt.xticks(Ks)
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Analysis for KMeans")
plt.grid(True)
plt.show()

# Select best k
best_k_sil = Ks[np.argmax(sil_scores)]
print(f"Best k from Silhouette: {best_k_sil}")

FINAL_K = best_k_sil  # or use best_k_elbow
print(f"\nUsing FINAL_K = {FINAL_K}")


* The Elbow Method shows the strongest curvature reduction around k = 4, indicating that adding more clusters beyond this point yields diminishing returns in inertia.
* The Distortion Score Elbow also identifies k = 4 as the optimal balance between model complexity and fit quality.
* However, Silhouette Scores increase steadily with more clusters and reach their highest value at k = 11, reflecting extremely clean geometric separation in latitude/longitude space.
* Validation metrics (Davies–Bouldin and Calinski–Harabasz) similarly improve as k increases, consistent with geographic partitioning patterns.
* Based on silhouette optimization, the model selects FINAL_K = 11, though this likely reflects spatial grouping rather than semantic business clusters.

In [ ]:
from sklearn.cluster import KMeans, MiniBatchKMeans
Ks = range(2, 12)
inertia = []
sil_scores = []
db_scores = []
ch_scores = []

print("\nEvaluating cluster quality across K...")

for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sample)

    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sample, labels))
    db_scores.append(davies_bouldin_score(X_sample, labels))
    ch_scores.append(calinski_harabasz_score(X_sample, labels))

    print(f"K={k} done.")

# ------------------------------------------------------------
# ELBOW PLOT
# ------------------------------------------------------------
plt.figure(figsize=(8,5))
plt.plot(Ks, inertia, marker='o')
plt.title("Elbow Method on Sample")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.grid(True)
plt.show()

# ------------------------------------------------------------
# METRIC PLOTS
# ------------------------------------------------------------
plt.figure(figsize=(10,5))
plt.plot(Ks, sil_scores, marker='o')
plt.title("Silhouette Score vs k")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.grid(True)
plt.show()

metrics_df = pd.DataFrame({
    'k': Ks,
    'silhouette': sil_scores,
    'davies_bouldin': db_scores,
    'calinski_harabasz': ch_scores
})
print("\nCluster validation metrics:")
print(metrics_df)

best_k_sil = Ks[np.argmax(sil_scores)]
print(f"\nBest k from silhouette = {best_k_sil}")

FINAL_K = best_k_sil
print(f"Using FINAL_K = {FINAL_K}")


print(f"\nTraining MiniBatchKMeans (k={FINAL_K}) on full dataset...")

kmeans = MiniBatchKMeans(
    n_clusters=FINAL_K,
    random_state=42,
    batch_size=1000,
    n_init=10
)

df_geo['cluster'] = kmeans.fit_predict(X_geo_scaled)
print("✓ Final clustering complete!")

* The Elbow Method shows a strong drop in inertia up to k = 4, after which improvements flatten, indicating diminishing returns beyond four clusters.
* Silhouette scores steadily increase with larger k values, reaching the highest value at k = 11, driven by strong geometric separation in latitude–longitude space.
* Cluster validation metrics (Davies–Bouldin ↓ and Calinski–Harabasz ↑) improve consistently with larger k, reinforcing the silhouette trend.
* Based on silhouette optimization, the model selects FINAL_K = 11, reflecting clean spatial partitioning rather than semantic grouping.

#Silhouette Analysis For Final Model

In [ ]:
from sklearn.metrics import silhouette_samples, silhouette_score
import numpy as np
import matplotlib.pyplot as plt

print("\nComputing silhouette values on sample ...")

# Predict labels for the sample using the FINAL trained MiniBatchKMeans model
labels_sample = kmeans.predict(X_sample)

# Compute silhouette values
sample_sil_vals = silhouette_samples(X_sample, labels_sample)
avg_sil = sample_sil_vals.mean()

print(f"Average silhouette (sample) = {avg_sil:.4f}")


plt.figure(figsize=(10, 7))

y_lower = 10

for c in range(FINAL_K):

    vals_c = sample_sil_vals[labels_sample == c]
    vals_c.sort()

    size_c = len(vals_c)
    y_upper = y_lower + size_c

    plt.fill_betweenx(
        np.arange(y_lower, y_upper),
        0,
        vals_c,
        alpha=0.7,
        label=None
    )

    plt.text(-0.05, (y_lower + y_upper) / 2, str(c))

    y_lower = y_upper + 10

plt.axvline(avg_sil, color="red", linestyle="--", label="Average Silhouette")

plt.xlabel("Silhouette coefficient")
plt.ylabel("Cluster")
plt.title(f"Silhouette Plot for k={FINAL_K}")
plt.legend()
plt.show()


* The average silhouette score is 0.9572, indicating extremely strong cluster separation.
* All clusters show consistently high silhouette values, reflecting clean geometric partitioning in the latitude–longitude feature space.
* The nearly uniform and wide silhouette bands suggest that K-Means is dividing the data into well-defined spatial regions rather than overlapping clusters.
* Such high scores confirm that the clustering structure is dominated by geographic separation, not business characteristics.

In [ ]:
for i in range(FINAL_K):
    print(f"\nCluster {i} (n={len(df_geo[df_geo['cluster']==i])}):")
    print(df_geo[df_geo['cluster']==i]['city'].value_counts().head(5))

* Clusters clearly represent geographic city regions.

In [ ]:
plt.figure(figsize=(14,8))
scatter = plt.scatter(df_geo['longitude'], df_geo['latitude'],
                     c=df_geo['cluster'], cmap='tab20',
                     s=5, alpha=0.6)
plt.colorbar(scatter, label='Cluster')
plt.title(f'Geographic Distribution of {FINAL_K} Clusters')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, alpha=0.3)
plt.show()

* The 11 clusters form clearly separated geographic regions, with each cluster centered around a distinct U.S. metro area. This confirms that K-Means is partitioning businesses primarily based on longitude–latitude location.

In [ ]:
# Cluster summary
cluster_summary = df_geo.groupby('cluster').agg({
    'business_id': 'count',
    'stars': 'mean',
    'review_count': 'mean',
    'city': lambda x: x.value_counts().index[0]  # Most common city
}).round(2)

cluster_summary.columns = ['n_businesses', 'avg_stars', 'avg_reviews', 'primary_city']
cluster_summary = cluster_summary.sort_values('n_businesses', ascending=False)
print(cluster_summary)

* Clusters correspond to major metro areas, with Philadelphia, Tampa, St. Louis, Nashville, and Indianapolis being the largest. Average star ratings are similar across clusters, while review counts vary, reflecting different levels of business activity per region.

# 5. DBSCAN Clustering

In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np

# Prepare coordinates in radians
coords = np.radians(df_geo[['latitude', 'longitude']].values)

# eps = distance in radians → 5 km ≈ 5 / 6371
eps_km = 5
eps = eps_km / 6371.0

db = DBSCAN(
    eps=eps,
    min_samples=50,
    metric='haversine'
)

df_geo['cluster_db'] = db.fit_predict(coords)

print("DBSCAN clusters found:", df_geo['cluster_db'].nunique())
print(df_geo['cluster_db'].value_counts().head())


* Found 18 clusters using geographic density.
* Largest groups correspond to major metro areas.
* DBSCAN effectively separates dense city regions and identifies sparse points as noise.

In [ ]:
# Check for noise points
noise_count = (df_geo['cluster_db'] == -1).sum()
print(f"\nNoise points (outliers): {noise_count}")

# Analyze each DBSCAN cluster
print("\n" + "="*60)
print("DBSCAN Cluster Analysis:")
print("="*60)

for cluster_id in sorted(df_geo['cluster_db'].unique()):
    if cluster_id == -1:
        print(f"\nNoise/Outliers (n={len(df_geo[df_geo['cluster_db']==-1])}):")
        print(df_geo[df_geo['cluster_db']==-1]['city'].value_counts().head(5))
    else:
        cluster_data = df_geo[df_geo['cluster_db']==cluster_id]
        print(f"\nCluster {cluster_id} (n={len(cluster_data)}):")
        print(cluster_data['city'].value_counts().head(3))

# Compare cluster assignments
comparison = pd.DataFrame({
    'kmeans': df_geo['cluster'],
    'dbscan': df_geo['cluster_db']
})
print("\n" + "="*60)
print("How many DBSCAN clusters overlap with each KMeans cluster?")
print("="*60)
print(comparison.groupby('kmeans')['dbscan'].nunique().sort_values(ascending=False))

* DBSCAN found 18 meaningful geographic clusters plus 953 outlier points.
* Each cluster corresponds to a distinct metro area (e.g., Santa Barbara, St. Louis, Tucson, Philadelphia, Tampa, Nashville, Indianapolis, New Orleans, Edmonton, Reno, Boise).
* Smaller clusters also capture local subregions such as Alton, Plant City, Green Valley, and Springfield.
* Comparison with K-Means shows that several DBSCAN metro clusters merge into a single K-Means region, confirming that DBSCAN provides more granular, density-based structure.

#6. HDBSCAN Clustering

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    metric='haversine',
    min_cluster_size=50,
    min_samples=20
)

df_geo['cluster_hdb'] = clusterer.fit_predict(coords)

print("HDBSCAN clusters:", df_geo['cluster_hdb'].nunique())
print(df_geo['cluster_hdb'].value_counts().head())


* HDBSCAN discovered 729 fine-grained clusters, showing very detailed geographic structure.
* A large number of points (46,329) were labeled as noise due to sparse or isolated locations.
* The largest meaningful clusters contain around 2,000–4,000 businesses each, reflecting dense metro regions.

#7. Geospatial Visualizations

In [ ]:
pip install folium

In [ ]:
import folium
from folium.plugins import MarkerCluster

cluster_label_column = 'cluster_hdb'
# Center map at median coordinate
m = folium.Map(
    location=[df_geo['latitude'].median(), df_geo['longitude'].median()],
    zoom_start=4
)

marker_cluster = MarkerCluster().add_to(m)

for _, row in df_geo.iterrows():
    label = row[cluster_label_column]

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color='blue' if label != -1 else 'red',
        fill=True,
        fill_opacity=0.6
    ).add_to(marker_cluster)

m


* Clusters appear as clear metro hotspots across the U.S.
* Largest density centers include Philadelphia, Tampa, Los Angeles, Nashville, Indianapolis, New Orleans, Tucson, and Boise.
* The visualization confirms DBSCAN captures true geographic population centers rather than forcing artificial partitions

In [ ]:
plt.figure(figsize=(10,6))
plt.hexbin(df_geo['longitude'], df_geo['latitude'], gridsize=100, cmap='inferno')
plt.colorbar(label='density')
plt.title("Geographic Density Heatmap")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


* Density is concentrated in a few major metro regions.
* Eastern U.S., Florida, California, and the Southwest show the highest business clusters.
* Most areas outside major cities have very low or zero density, reflecting sparse Yelp coverage.

#8. Dimensionality Reduction (UMAP)

In [ ]:
pip install umap-learn


In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -----------------------------------------------------------
# Prepare coordinates in radians for haversine UMAP
# -----------------------------------------------------------
coords_rad = np.radians(df_geo[['latitude', 'longitude']].values)

umap_model = umap.UMAP(
    n_neighbors=50,
    min_dist=0.15,
    n_components=2,
    metric='haversine',
    random_state=42,
)

embedding = umap_model.fit_transform(coords_rad)

# -----------------------------------------------------------
# Build color map safely
# -----------------------------------------------------------
cluster_labels = df_geo['cluster_hdb']

# Exclude noise (-1)
valid_clusters = sorted([c for c in cluster_labels.unique() if c != -1])
num_clusters = len(valid_clusters)

# Create palette with correct number of colors
palette = sns.color_palette("Spectral", num_clusters)

# Map cluster IDs -> palette colors
color_map = {cid: palette[i] for i, cid in enumerate(valid_clusters)}

# Noise = gray
color_map[-1] = (0.5, 0.5, 0.5)

# Convert labels -> colors
point_colors = cluster_labels.map(color_map)

# -----------------------------------------------------------
# Plot UMAP
# -----------------------------------------------------------
plt.figure(figsize=(12, 7))
plt.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=point_colors,
    s=4,
    alpha=0.7,
    linewidths=0
)

plt.title("HDBSCAN Cluster Structure (UMAP Projection)", fontsize=14)
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")

plt.tight_layout()
plt.show()


* UMAP reveals many small, distinct HDBSCAN clusters spread across the manifold.
* Curved, ribbon-like shapes reflect local geographic neighborhoods preserved in reduced dimensions.
* Noise points (gray) are scattered throughout, consistent with sparse or isolated locations.

#**Key Insights**

* Yelp businesses cluster strongly by geographic metro areas.
* DBSCAN captures realistic city boundaries better than K-Means.
* Star ratings are mostly high and review counts follow a long tail.
* A few categories (e.g., Restaurants, Food, Shopping) dominate the dataset.
* *UMAP* confirms clusters reflect location, not business attributes.

**Congratulation**!